(course-enzyme-openmm-integration)=

# Path B - Module 45: OpenMM Integration

It's time to put your engineered PETase to the test. To see if your new disulfide bridge resists the industrial heat, you need to run a **Molecular Dynamics** simulation. 

In this module, you will learn to convert your MolSys object into a high-performance **OpenMM Simulation** to observe the enzyme in motion.

In [ ]:
import molsysmt as msm
from molsysmt import systems

# Load our solvated and minimized complex
petase = msm.convert('pdb:6EQE', to_form='molsysmt.MolSys', selection='molecule_type=="protein"')
msm.build.add_missing_hydrogens(petase, pH=8.0)
bhet = msm.convert('C1=CC(=CC=C1C(=O)OCCO)C(=O)OCCO', from_form='string:smiles', to_form='molsysmt.MolSys')
molsys = msm.merge([petase, bhet])
msm.molecular_mechanics.potential_energy_minimization(molsys)

### 1. Initializing the Reactor (Simulation Object)
Using `msm.convert()`, you can generate the three pillars of an OpenMM simulation: the **System**, the **Integrator**, and the **Simulation** object.

In [ ]:
# Convert to OpenMM Simulation
# Setting the temperature to 343 K (70 degrees Celsius) to simulate industrial heat
try:
    simulation = msm.convert(molsys, to_form='openmm.Simulation', 
                             forcefield='amber14-all.xml', water_model='amber14/tip3pfb.xml')
    print(f"OpenMM Simulation ready: {type(simulation)}")
except Exception as e:
    print(f"OpenMM initialization failed: {e}")

### 2. Running the Industrial Test
Let's run a very short production run (500 steps) to see if the enzyme maintains its integrity at 70°C.

In [ ]:
if 'simulation' in locals():
    print("Starting 500 steps of High-Temperature MD...")
    simulation.step(500)
    print("Simulation completed.")

### 3. Extracting the Relaxed State
After the simulation, we pull the coordinates back to MolSysMT to visualize the final pose of the plastic substrate.

In [ ]:
if 'simulation' in locals():
    # Update coordinates from the simulation context
    new_positions = simulation.context.getState(getPositions=True).getPositions()
    msm.set(molsys, coordinates=new_positions)
    
    # View the final result
    msm.view(molsys)

--- 

### 🏆 Path B Challenge: The Thermal Tester

1. Take your **Mutant PETase** with the added disulfide bridge.
2. Setup an OpenMM simulation at **400 K** (very extreme heat).
3. Run **1000 steps**.
4. Calculate the **RMSD** of the protein. Does it unfold? (A high RMSD > 0.5 nm indicates unfolding).

You have successfully simulated your industrial catalyst! In **Module 46**, we will learn how to use **Geometric Transformations** to orient the enzyme in the box for optimal analysis.